In [1]:
from dataclasses import dataclass
from typing import Any, Callable, TypedDict

from dotenv import load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse, ToolCallRequest, dynamic_prompt, wrap_model_call, wrap_tool_call
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import tool
from langchain_mistralai import ChatMistralAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from pydantic import BaseModel

load_dotenv()

True

# Dynamic Model Selection

In [7]:
small_model= ChatMistralAI(model="mistral-small-latest")
large_model= ChatMistralAI(model="mistral-medium-latest")

In [8]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """ Choose model based on conversation or query complexity. """
    message_count= len(request.state['messages'])
    
    if message_count> 3:
        model= large_model
    else:
        model= small_model
    
    return handler(request.override(model=model))

checkpointer= InMemorySaver()

agent = create_agent(
    model=small_model,  # Default model
    middleware=[dynamic_model_selection],
    checkpointer= checkpointer
)

In [9]:
message= HumanMessage(content="Answer in one word only for every question.")
config= {"configurable": {"thread_id": "1"}}
response= agent.invoke({"messages": [message]}, config= config)
print(response['messages'][-1].content)
print('-'*50)
print(response['messages'][-1].response_metadata['model_name'])

print('='*80)

message= HumanMessage(content="What is the square root of 4?")
config= {"configurable": {"thread_id": "1"}}
response= agent.invoke({"messages": [message]}, config= config)
print(response['messages'][-1].content)
print('-'*50)
print(response['messages'][-1].response_metadata['model_name'])

print('='*80)

message= HumanMessage(content="What is the capital of Germany?")
config= {"configurable": {"thread_id": "1"}}
response= agent.invoke({"messages": [message]}, config= config)
print(response['messages'][-1].content)
print('-'*50)
print(response['messages'][-1].response_metadata['model_name'])

print('='*80)

message= HumanMessage(content="Who is Michael Jordan?")
config= {"configurable": {"thread_id": "1"}}
response= agent.invoke({"messages": [message]}, config= config)
print(response['messages'][-1].content)
print('-'*50)
print(response['messages'][-1].response_metadata['model_name'])

Understood. Ask your questions, and I'll respond with one word each.
--------------------------------------------------
mistral-small-latest
Two
--------------------------------------------------
mistral-small-latest
Berlin.
--------------------------------------------------
mistral-medium-latest
Basketballer.
--------------------------------------------------
mistral-medium-latest


In [10]:
response

{'messages': [HumanMessage(content='Answer in one word only for every question.', additional_kwargs={}, response_metadata={}, id='84ab76dd-ce71-49d3-96d3-de65e059345a'),
  AIMessage(content="Understood. Ask your questions, and I'll respond with one word each.", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 12, 'total_tokens': 29, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019cd66d-19fb-7de3-a768-bb0d6cf7c76a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 17, 'total_tokens': 29}),
  HumanMessage(content='What is the square root of 4?', additional_kwargs={}, response_metadata={}, id='c3e857de-14aa-426c-937b-28a5c47ef8b2'),
  AIMessage(content='Two', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 40, 'total_tokens': 42, 

# Dynamic Tool

## State Based

In [69]:
@wrap_model_call
def state_based_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """ Filter tools based on conversation state """
    state= request.state
    is_authenticated= state.get("authenticated", True) # sample 
    message_count= len(state['messages'])
    
    if not is_authenticated:
        tools= [t for t in request.tools if t.name.startswith("public_")]
        request= request.override(tools=tools)
    elif message_count < 5:
        tools= [t for t in request.tools if t.name != "advance_search"]
        request= request.override(tools=tools)
    
    return handler(request)

In [70]:
@tool
def public_search(query: str) -> str:
    """ Use this tool for public search """
    return "Your query is being executed in public search"

@tool
def private_search(query: str) -> str:
    """ Use this tool for private search """
    return "Your query is being executed in private search"

@tool
def advance_search(query: str) -> str:
    """ Use this tool for advance search """
    return "Your query is being executed in advance search"

In [71]:
model= ChatMistralAI(model="mistral-medium-latest")
agent = create_agent(
    model=model,
    tools= [public_search, private_search, advance_search],
    middleware=[state_based_tools],
    system_prompt='You are a helpful assistant. You have access to certain tools. Your respoonse should be based on the tool only. Do not answer anything from your own knowledge',
)

In [72]:
messages= []

In [79]:
query= HumanMessage(content='What tools do you have access to right now?')
messages.append(query)
response= agent.invoke({"messages": messages})
messages.append(response['messages'][-1])

In [80]:
response # advance tool is available after 5 messages

{'messages': [HumanMessage(content='What tools do you have access to right now?', additional_kwargs={}, response_metadata={}, id='6f9022cd-6f8d-4d92-8b47-b36be29e68fb'),
  AIMessage(content='I currently have access to the following tools:\n\n1. **Public Search**: Useful for searching publicly available information based on a query.\n2. **Private Search**: Useful for searching private or restricted information based on a query.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 163, 'total_tokens': 210, 'completion_tokens': 47, 'prompt_tokens_details': {'cached_tokens': 128}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019cd69a-0e86-7d40-9cf7-6645a919d56e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 163, 'output_tokens': 47, 'total_tokens': 210}),
  HumanMessage(content='What tools do you have access to right now?', additional_kwargs={}

## Store Based

In [111]:
@dataclass
class Context:
    user_id: str

@wrap_model_call
def store_based_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Filter tools based on Store preferences."""
    user_id = request.runtime.context.user_id

    store = request.runtime.store
    feature_flags = store.get(("features",), user_id)

    if feature_flags:
        enabled_features = feature_flags.value.get("enabled_tools", [])

        tools = [t for t in request.tools if t.name in enabled_features]
    else:
        tools= []
    request = request.override(tools=tools)

    return handler(request)


In [112]:
@tool
def public_search(query: str) -> str:
    """ Use this tool for public search """
    return "Your query is being executed in public search"

@tool
def private_search(query: str) -> str:
    """ Use this tool for private search """
    return "Your query is being executed in private search"

@tool
def advance_search(query: str) -> str:
    """ Use this tool for advance search """
    return "Your query is being executed in advance search"

In [113]:
model= ChatMistralAI(model="mistral-medium-latest")
agent= create_agent(
    model= model,
    tools= [public_search, private_search, advance_search],
    middleware=[store_based_tools],
    context_schema=Context,
    store=InMemoryStore(),
    system_prompt='You are a helpful assistant. You have access to certain tools. Your respoonse should be based on the tool only. Do not answer anything from your own knowledge',
)

In [114]:
store = agent.store

store.put(
    ("features",),
    "user_1",
    {"enabled_tools": ["public_search"]}
)

store.put(
    ("features",),
    "user_2",
    {"enabled_tools": ["public_search", "private_search"]}
)

store.put(
    ("features",),
    "user_3",
    {"enabled_tools": ["public_search", "private_search", "advance_search"]}
)

In [115]:
query= HumanMessage(content='What tools do you have access to right now?')
response= agent.invoke({"messages": [query]}, context=Context(user_id="user_4"))
response

{'messages': [HumanMessage(content='What tools do you have access to right now?', additional_kwargs={}, response_metadata={}, id='af1df3ea-2e14-4b99-80bb-6a0fac199e17'),
  AIMessage(content='Currently, I have access to the following tools:\n\n1. **Multi Tool Use (Parallel Function Calling)** - Allows me to use multiple tools at once if needed.\n\n### **Search and Information Tools**\n2. **Google Search** - For retrieving up-to-date information from the web.\n3. **Wikipedia Search** - For fetching structured knowledge from Wikipedia.\n4. **YouTube Search** - For finding relevant videos on YouTube.\n5. **News API** - For fetching the latest news articles.\n\n### **Coding and Development Tools**\n6. **Python Code Interpreter** - Executes Python code in a secure environment.\n7. **JavaScript Code Interpreter** - Runs JavaScript code snippets.\n8. **Bash Terminal** - Executes shell commands (limited to safe operations).\n9. **GitHub Repository Search** - Finds and retrieves GitHub repositor

## Runtime Context Based

In [129]:
@dataclass
class Context:
    user_role: str

@wrap_model_call
def context_based_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Filter tools based on Runtime Context permissions."""
    if request.runtime is None or request.runtime.context is None:
        user_role = "viewer"
    else:
        user_role = request.runtime.context.user_role

    if user_role == "admin":
        pass
    elif user_role == "editor":
        tools = [t for t in request.tools if t.name != "advance_search"]
        request = request.override(tools=tools)
    else:
        tools = [t for t in request.tools if t.name.startswith("public_")]
        request = request.override(tools=tools)

    return handler(request)

In [130]:
@tool
def public_search(query: str) -> str:
    """ Use this tool for public search """
    return "Your query is being executed in public search"

@tool
def private_search(query: str) -> str:
    """ Use this tool for private search """
    return "Your query is being executed in private search"

@tool
def advance_search(query: str) -> str:
    """ Use this tool for advance search """
    return "Your query is being executed in advance search"

In [131]:
model= ChatMistralAI(model="mistral-medium-latest")
agent= create_agent(
    model= model,
    tools= [public_search, private_search, advance_search],
    middleware=[context_based_tools],
    context_schema=Context,
    system_prompt='You are a helpful assistant. You have access to certain tools. Your respoonse should be based on the tool only. Do not answer anything from your own knowledge',
)

In [133]:
query= HumanMessage(content='What tools do you have access to right now?')
response= agent.invoke({"messages": [query]}, context=Context(user_role=""))
response

{'messages': [HumanMessage(content='What tools do you have access to right now?', additional_kwargs={}, response_metadata={}, id='97930560-1177-460c-81e8-2bd057eb5400'),
  AIMessage(content='I currently have access to the following tool:\n\n1. **Public Search Tool**: This tool allows me to perform public searches on the internet to fetch information based on your query.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 107, 'total_tokens': 143, 'completion_tokens': 36, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019cd6bd-ffef-7323-b0ed-a2d31c5ecaea-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 107, 'output_tokens': 36, 'total_tokens': 143})]}

## Runtime Tool Registration

In [140]:
@tool
def get_weather():
    """ Return weather information. """
    return 'The weather is goood.'

@tool
def calculate_tip(bill_amount: float, tip_percentage: float = 20.0) -> str:
    """Calculate the tip amount for a bill."""
    tip = bill_amount * (tip_percentage / 100)
    return f"Tip: ${tip:.2f}, Total: ${bill_amount + tip:.2f}"

class DynamicToolMiddleware(AgentMiddleware):
    """Middleware that registers and handles dynamic tools."""

    def wrap_model_call(self, request: ModelRequest, handler):
        updated = request.override(tools=[*request.tools, calculate_tip])
        return handler(updated)

    def wrap_tool_call(self, request: ToolCallRequest, handler):
        if request.tool_call["name"] == "calculate_tip":
            return handler(request.override(tool=calculate_tip))
        return handler(request)


In [141]:
model= ChatMistralAI(model="mistral-medium-latest")

agent = create_agent(
    model=model,
    tools=[get_weather],  
    middleware=[DynamicToolMiddleware()],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Calculate a 20% tip on $85"}]
})

In [142]:
result

{'messages': [HumanMessage(content='Calculate a 20% tip on $85', additional_kwargs={}, response_metadata={}, id='b9be96de-861c-427b-b55f-220c2cfd3aaf'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'BGQlVWwcZ', 'function': {'name': 'calculate_tip', 'arguments': '{"bill_amount": 85, "tip_percentage": 20}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 136, 'total_tokens': 159, 'completion_tokens': 23, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cd6f9-4877-7252-a340-d3c9012dfccd-0', tool_calls=[{'name': 'calculate_tip', 'args': {'bill_amount': 85, 'tip_percentage': 20}, 'id': 'BGQlVWwcZ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 136, 'output_tokens': 23, 'total_tokens': 159}),
  ToolMessage(content='Tip: $17.00, Total: $102.00', name='calculate_tip', id='517

# Tool Error Handling

In [143]:
@tool
def divisor(a: float, b: float) -> float:
    """ Use this to divide first number a by second number b. """
    return a / b

In [148]:
@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(content=f"Tool error: Please check your input and try again. ({str(e)})", tool_call_id=request.tool_call["id"])

In [151]:
model= ChatMistralAI(model="mistral-medium-latest")
agent = create_agent(
    model= model,
    tools=[divisor],
    middleware=[handle_tool_errors],
    system_prompt='You are a helpful assistant. You have access to certain tools. Your respoonse should be based on the tool only. Do not answer anything from your own knowledge',
)

In [152]:
query= HumanMessage(content='Divide the following two numbers: 10 and 0')
response= agent.invoke({"messages": [query]})
response

{'messages': [HumanMessage(content='Divide the following two numbers: 10 and 0', additional_kwargs={}, response_metadata={}, id='8ca37039-42e0-434b-ba4f-cd0cbfe8c69e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kaBuAdIco', 'function': {'name': 'divisor', 'arguments': '{"a": 10, "b": 0}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 128, 'total_tokens': 147, 'completion_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cd704-1f17-7460-97ab-d5c82e62d7d2-0', tool_calls=[{'name': 'divisor', 'args': {'a': 10, 'b': 0}, 'id': 'kaBuAdIco', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 128, 'output_tokens': 19, 'total_tokens': 147}),
  ToolMessage(content='Tool error: Please check your input and try again. (float division by zero)', id='7cc6e818-9822-4860-b8

# Dynamic System Prompt

In [165]:
class Context(TypedDict):
    user_role: str
    
@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} answer in one word only."

    return base_prompt

In [166]:
model= ChatMistralAI(model="mistral-medium-latest")
agent = create_agent(
    model= model,
    middleware=[user_role_prompt],
    context_schema=Context
)

In [168]:
message= HumanMessage(content="What is the capital of India")

response= agent.invoke({"messages": [message]}, context=Context(user_role="expert"))

response

{'messages': [HumanMessage(content='What is the capital of India', additional_kwargs={}, response_metadata={}, id='e83bdb65-c7ad-4752-8f79-82a8eaedafd7'),
  AIMessage(content='The capital of **India** is **New Delhi**, a metropolitan city within the larger **National Capital Territory (NCT) of Delhi**.\n\n### **Key Details About New Delhi:**\n1. **Geographical Location**:\n   - Latitude: **28.6139° N**\n   - Longitude: **77.2090° E**\n   - Elevation: ~216 meters (709 ft) above sea level\n   - Located on the banks of the **Yamuna River** (a tributary of the Ganges).\n\n2. **Administrative Status**:\n   - **New Delhi** is a **municipal district** within the **NCT of Delhi**, which is a **Union Territory** (not a state) with partial statehood.\n   - The **President of India** appoints the **Lieutenant Governor (LG)** of Delhi, while the **Chief Minister (CM)** heads the elected government.\n   - The **New Delhi Municipal Council (NDMC)** governs the capital city proper.\n\n3. **Historical

In [170]:
print(response['messages'][-1].content)

The capital of **India** is **New Delhi**, a metropolitan city within the larger **National Capital Territory (NCT) of Delhi**.

### **Key Details About New Delhi:**
1. **Geographical Location**:
   - Latitude: **28.6139° N**
   - Longitude: **77.2090° E**
   - Elevation: ~216 meters (709 ft) above sea level
   - Located on the banks of the **Yamuna River** (a tributary of the Ganges).

2. **Administrative Status**:
   - **New Delhi** is a **municipal district** within the **NCT of Delhi**, which is a **Union Territory** (not a state) with partial statehood.
   - The **President of India** appoints the **Lieutenant Governor (LG)** of Delhi, while the **Chief Minister (CM)** heads the elected government.
   - The **New Delhi Municipal Council (NDMC)** governs the capital city proper.

3. **Historical Background**:
   - Established as the capital of British India in **1911** (replacing **Calcutta/Kolkata**).
   - Designed by British architects **Edwin Lutyens** and **Herbert Baker** (hen

# Structured Output

In [172]:
class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str
    
model= ChatMistralAI(model="mistral-medium-latest")
agent = create_agent(
    model= model,
    response_format=ToolStrategy(ContactInfo)
)

In [173]:
query= HumanMessage(content='Extract the contact information from the following text: My name is John Doe, my email is 0q0cA@example.com, and my phone number is 123-456-7890.')
response= agent.invoke({"messages": [query]})
response

{'messages': [HumanMessage(content='Extract the contact information from the following text: My name is John Doe, my email is 0q0cA@example.com, and my phone number is 123-456-7890.', additional_kwargs={}, response_metadata={}, id='e4c4c935-d2c9-462d-8b87-9d10bfb43fd3'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'S7I6APcic', 'function': {'name': 'ContactInfo', 'arguments': '{"name": "John Doe", "email": "0q0cA@example.com", "phone": "123-456-7890"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 125, 'total_tokens': 166, 'completion_tokens': 41, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cd730-c205-71e1-bc14-babc182398cb-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email': '0q0cA@example.com', 'phone': '123-456-7890'}, 'id': 'S7I6APcic', 'type': 'tool_call'}], inv

In [174]:
response['structured_response']

ContactInfo(name='John Doe', email='0q0cA@example.com', phone='123-456-7890')

In [175]:
query= HumanMessage(content='Hello')
response= agent.invoke({"messages": [query]})
response

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='e5ed913f-a445-47aa-a3dd-682e2ff3fe9f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'KC0KufRNs', 'function': {'name': 'ContactInfo', 'arguments': '{"name": "Unknown", "email": "", "phone": ""}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 80, 'total_tokens': 100, 'completion_tokens': 20, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cd733-5dbc-70a3-9284-e3e4c5adf415-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'Unknown', 'email': '', 'phone': ''}, 'id': 'KC0KufRNs', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 80, 'output_tokens': 20, 'total_tokens': 100}),
  ToolMessage(content="Returning structured response: name='Unknown' email='' phone=''", name='Contac

# Memory

## Using Middleware

In [227]:
@tool
def Tool1(text: str) -> str:
    """ This tool converts text to uppercase """
    return text.upper()

@tool
def Tool2(text: str) -> str:
    """ This tool converts text to lowercase """
    return text.lower()

In [228]:
class CustomState(AgentState):
    user_preferences: dict

class CustomMiddelware(AgentMiddleware):
    state_schema= CustomState
    tools= [Tool1, Tool2]
    
    def before_model(self, state: CustomState, runtime):
        """ This part runs before model call, We can modify the prompt here. """
        prefs= state.get("user_preferences")
        
        if not prefs:
            return None
        
        style= prefs.get("style", "normal")
        verbosity= prefs.get("verbosity", "short")
        
        system_prompt = (
            f"You are an assistant. "
            f"Respond in {style} style with {verbosity} explanation."
        )

        messages = state["messages"]

        new_messages = [{"role": "system", "content": system_prompt}] + messages

        return {
            "messages": new_messages
        }

In [229]:
model= ChatMistralAI(model="mistral-medium-latest")
agent = create_agent(
    model= model,
    tools= [Tool1, Tool2],
    middleware=[CustomMiddelware()]
)

In [230]:
results= agent.invoke({
    "messages": [
        {"role": "user", "content": "Explain what Python decorators are"}
    ],
    "user_preferences": {
        "style": "rude",
        "verbosity": "very short"
    }
})

results

{'messages': [HumanMessage(content='Explain what Python decorators are', additional_kwargs={}, response_metadata={}, id='fe23996d-f58a-4f7f-befd-9d3577911858'),
  SystemMessage(content='You are an assistant. Respond in rude style with very short explanation.', additional_kwargs={}, response_metadata={}, id='21d6e0bc-529d-44d0-9704-6f2c5c3e9f29'),
  AIMessage(content='Decorators in Python are functions that modify other functions. They use `@decorator_name` syntax. Done.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 141, 'total_tokens': 165, 'completion_tokens': 24, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019cd779-422f-7ff1-a0fc-c169dd5c9c83-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 24, 'total_tokens': 165})],
 'user_preferences': {'style': 'rude', 'verbosi

In [231]:
print(results['messages'][-1].content)

Decorators in Python are functions that modify other functions. They use `@decorator_name` syntax. Done.


## Using State Schema

In [2]:
class CustomState(AgentState):
    user_preferences: dict
    
model= ChatMistralAI(model="mistral-medium-latest")
agent = create_agent(
    model= model,
    state_schema= CustomState
)

In [3]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "I want you to explain OOP in python."}],
    "user_preferences": {"style": "funny", "verbosity": "very short"},
})
result

{'messages': [HumanMessage(content='I want you to explain OOP in python.', additional_kwargs={}, response_metadata={}, id='85681683-8e92-43f8-9c3b-f7717f8d6d01'),
  AIMessage(content='Object-Oriented Programming (OOP) in Python is a programming paradigm that organizes code into **objects**—self-contained units that combine **data (attributes)** and **behavior (methods)**. Python fully supports OOP, making it easier to model real-world entities and write modular, reusable code.\n\n---\n\n## **Core Concepts of OOP in Python**\n### 1. **Class**\nA **class** is a blueprint for creating objects. It defines attributes (data) and methods (functions) that the objects will have.\n\n```python\nclass Dog:\n    # Class attribute (shared by all instances)\n    species = "Canis familiaris"\n\n    # Initializer / Constructor\n    def __init__(self, name, age):\n        # Instance attributes (unique to each object)\n        self.name = name\n        self.age = age\n\n    # Method (function inside a cl

In [4]:
print(result['messages'][-1].content)

Object-Oriented Programming (OOP) in Python is a programming paradigm that organizes code into **objects**—self-contained units that combine **data (attributes)** and **behavior (methods)**. Python fully supports OOP, making it easier to model real-world entities and write modular, reusable code.

---

## **Core Concepts of OOP in Python**
### 1. **Class**
A **class** is a blueprint for creating objects. It defines attributes (data) and methods (functions) that the objects will have.

```python
class Dog:
    # Class attribute (shared by all instances)
    species = "Canis familiaris"

    # Initializer / Constructor
    def __init__(self, name, age):
        # Instance attributes (unique to each object)
        self.name = name
        self.age = age

    # Method (function inside a class)
    def bark(self):
        return f"{self.name} says woof!"
```

### 2. **Object (Instance)**
An **object** is an instance of a class. You create objects using the class name followed by parenthese

# Streaming

In [236]:
model= ChatMistralAI(model="mistral-medium-latest")
agent = create_agent(
    model= model
)

In [ ]:
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Search for AI news and summarize the findings"}]
}, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        if isinstance(latest_message, HumanMessage):
            print(f"User: {latest_message.content}")
        elif isinstance(latest_message, AIMessage):
            print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

User: Search for AI news and summarize the findings
Agent: Here’s a summary of the latest **AI news** (as of **June 2024**), covering breakthroughs, controversies, and industry trends:

---

### **🔥 Top AI Developments**
1. **OpenAI’s GPT-4o & Voice Mode Rollout**
   - OpenAI launched **GPT-4o** ("Omni"), a faster, cheaper model with **real-time voice and vision capabilities**.
   - The **advanced Voice Mode** (with emotional tone detection) is being rolled out to **plus users** after delays due to safety testing.
   - Critics raise concerns about **deepfake risks** and **job displacement** in customer service.

2. **Google’s AI Overhaul**
   - **Gemini 1.5 Flash**: A lightweight, cost-effective model for high-volume tasks (e.g., chatbots, data extraction).
   - **Project Astra**: A real-time multimodal AI assistant (demoed at Google I/O) that processes video, audio, and text simultaneously.
   - **AI-generated search results** now dominate Google queries, sparking debates over **accur